# 00 — Ingest & cache (Phase 1)

**Run-once rule:** this notebook is the ONLY code that touches the network.
It pulls each source into `data/raw/` exactly once; every downstream notebook
reads from `data/raw/` and never re-fetches. Each fetch cell below is guarded
by an `if not path.exists()` check so accidental re-runs are no-ops.

| Source | Target file | Notes |
|---|---|---|
| bechdeltest.com `getAllMovies` | `data/raw/bechdel_all_movies.json` | shared hosting, flaky — ONE pull, 300s timeout. CC BY-NC 3.0. |
| Pudding `character_list5.csv` | `data/raw/pudding_character_list5.csv` | `graphs` branch of matthewfdaniels/scripts |
| Pudding `meta_data7.csv` | `data/raw/pudding_meta_data7.csv` | same repo/branch |
| The Numbers — TidyTuesday snapshot (2018) | `data/raw/tidytuesday_movie_profit.csv` | 3,401 films, budgets ≤ $175M |
| The Numbers — WWU mirror (2023) | `data/raw/wwu_mirror_pages/all_*.html` | top 1,000 films by budget |

Full acquisition history and rationale: `data/raw/README_provenance.md`.

In [2]:
from pathlib import Path

import requests

RAW = Path.cwd().parent / "data" / "raw"
assert RAW.is_dir(), f"expected {RAW} to exist — launch jupyter from the project root or notebooks/"
RAW

PosixPath('/Users/trevorjmckee/Documents/Work/School/03_Summer_2026/DSC530/final_project/data/raw')

## 1. Bechdel test — full DB (single pull, cached)

**Server reliability note (2026-07-04/05):** the endpoint repeatedly failed
server-side — connection cut with an empty reply at ~60s, across 9+ attempts
over 14 hours (their gateway appears to kill the response before the full dump
finishes generating).

**RESOLVED 2026-07-12 — API permanently disabled; do not retry.** The
endpoint now returns HTTP 410 `{"description":"API has been disabled"}`
(the `/api/v1/doc` page too). The site is alive but offers no bulk export,
so the cell below can never succeed; it is kept as a record of the attempt.
The TidyTuesday 2021-03-09 mirror (next cell) is the FINAL source — see
`data/raw/README_provenance.md`.

In [ ]:
BECHDEL_URL = "https://bechdeltest.com/api/v1/getAllMovies"
bechdel_path = RAW / "bechdel_all_movies.json"

if bechdel_path.exists():
    print(f"cached — skipping fetch ({bechdel_path.stat().st_size:,} bytes)")
else:
    resp = requests.get(BECHDEL_URL, timeout=300)
    resp.raise_for_status()
    assert resp.content[:1] == b"[", "response is not a JSON array — server likely returned an error page"
    bechdel_path.write_bytes(resp.content)
    print(f"saved {bechdel_path.stat().st_size:,} bytes -> {bechdel_path}")

In [ ]:
# Fallback (ADOPTED 2026-07-05 while the API stays down): TidyTuesday's
# 2021-03-09 mirror of the same DB. If the live pull above ever succeeds,
# it supersedes this file — see data/raw/README_provenance.md.
BECHDEL_MIRROR_URL = ("https://raw.githubusercontent.com/rfordatascience/"
                      "tidytuesday/master/data/2021/2021-03-09/raw_bechdel.csv")
mirror_path = RAW / "bechdel_tidytuesday_2021.csv"
if mirror_path.exists():
    print(f"cached — skipping mirror ({mirror_path.stat().st_size:,} bytes)")
else:
    resp = requests.get(BECHDEL_MIRROR_URL, timeout=60)
    resp.raise_for_status()
    mirror_path.write_bytes(resp.content)
    print(f"saved {mirror_path.stat().st_size:,} bytes -> {mirror_path}")

## 2. Pudding film-dialogue files (`graphs` branch)

In [4]:
PUDDING_BASE = "https://raw.githubusercontent.com/matthewfdaniels/scripts/graphs"
for fname, target in [
    ("character_list5.csv", "pudding_character_list5.csv"),
    ("meta_data7.csv", "pudding_meta_data7.csv"),
]:
    path = RAW / target
    if path.exists():
        print(f"cached — skipping {fname}")
        continue
    resp = requests.get(f"{PUDDING_BASE}/{fname}", timeout=60)
    resp.raise_for_status()
    path.write_bytes(resp.content)
    print(f"saved {path.stat().st_size:,} bytes -> {path}")

saved 677,440 bytes -> /Users/trevorjmckee/Documents/Work/School/03_Summer_2026/DSC530/final_project/data/raw/pudding_character_list5.csv
saved 308,519 bytes -> /Users/trevorjmckee/Documents/Work/School/03_Summer_2026/DSC530/final_project/data/raw/pudding_meta_data7.csv


## 3. The Numbers — box office (two static mirrors)

**What happened to the original plan:** the intended source
(`the-numbers.com/movie/budgets/all`) no longer exists — on 2026-07-04 the
site served its homepage with HTTP 404 at that URL to *every* client,
including a real browser (verified via a saved page). A polite scrape is
therefore impossible, and two documented static mirrors of the same table
are used instead.

**UPDATE 2026-07-12 — root cause:** The Numbers announced a ground-up site
rebuild on 2026-03-11 ("The new The Numbers"); historical pages including the
budgets table are "under construction" and being restored gradually. The
table wasn't paywalled or deliberately removed — it just hasn't been rebuilt
yet (still absent as of 2026-07-12). Since the study window ends in 2015 and
both mirrors post-date all grosses in it, a restored live table would add
nothing; the mirror union is the final source. Details in
`data/raw/README_provenance.md`.

1. **TidyTuesday `movie_profit.csv`** (scraped Oct 2018) — 3,401 films with
   release_date, movie, production_budget, domestic/worldwide gross (nominal),
   plus distributor, mpaa_rating, genre. Deep small-budget coverage, but
   **top-truncated at $175M budgets**.
2. **WWU DATA 311 mirror** (early-2023 snapshot) — the top 1,000 films by
   production budget as 10 paginated HTML pages. Covers exactly the mega-budget
   films TidyTuesday lacks.

They are **unioned** (dedupe on normalized title + year) in
`02_transform_numbers.ipynb` → `data/interim/the_numbers_union.csv`.
Since the study window ends in 2015, grosses were final at both snapshot
dates — the vintage difference doesn't affect our films. Both grosses are
NOMINAL dollars (never mix with Pudding's inflation-adjusted gross).

In [ ]:
import time

# --- mirror 1: TidyTuesday snapshot (GitHub, static) ---
TT_URL = "https://raw.githubusercontent.com/rfordatascience/tidytuesday/master/data/2018/2018-10-23/movie_profit.csv"
tt_path = RAW / "tidytuesday_movie_profit.csv"
if tt_path.exists():
    print(f"cached — skipping TidyTuesday ({tt_path.stat().st_size:,} bytes)")
else:
    resp = requests.get(TT_URL, timeout=60)
    resp.raise_for_status()
    tt_path.write_bytes(resp.content)
    print(f"saved {tt_path.stat().st_size:,} bytes -> {tt_path}")

# --- mirror 2: WWU pages (static university hosting; throttled anyway) ---
WWU_BASE = ("https://facultyweb.cs.wwu.edu/~wehrwes/courses/data311_23w/"
            "lab5/the-numbers.com/movie/budgets/all")
wwu_dir = RAW / "wwu_mirror_pages"
wwu_dir.mkdir(exist_ok=True)
for off in [1, 101, 201, 301, 401, 501, 601, 701, 801, 901]:
    path = wwu_dir / f"all_{off:04d}.html"
    if path.exists():
        print(f"cached — skipping {path.name}")
        continue
    url = f"{WWU_BASE}/" if off == 1 else f"{WWU_BASE}/{off}/"
    resp = requests.get(url, timeout=60)
    resp.raise_for_status()
    path.write_bytes(resp.content)
    print(f"saved {path.stat().st_size:,} bytes -> {path.name}")
    time.sleep(1.5)

## 4. Verify the cache

Quick sanity pass — files exist, JSON parses, CSVs have plausible row counts. No cleaning here; that's Phase 2+.

In [ ]:
import json

import pandas as pd

if bechdel_path.exists():
    movies = json.loads(bechdel_path.read_text())
    print(f"bechdel: {len(movies):,} films; sample record: {movies[0]}")
else:
    print("bechdel: NOT YET ACQUIRED (server still failing) — re-run cell 1 another day")

for target in ["pudding_character_list5.csv", "pudding_meta_data7.csv"]:
    df = pd.read_csv(RAW / target, encoding="latin-1")
    print(f"{target}: {df.shape[0]:,} rows x {df.shape[1]} cols")

tt = pd.read_csv(RAW / "tidytuesday_movie_profit.csv", index_col=0)
print(f"tidytuesday_movie_profit.csv: {tt.shape[0]:,} rows x {tt.shape[1]} cols")

wwu_pages = sorted((RAW / "wwu_mirror_pages").glob("all_*.html"))
print(f"wwu_mirror_pages: {len(wwu_pages)} pages "
      f"({sum(p.stat().st_size for p in wwu_pages):,} bytes total)")